In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 52 — Structured JSON Extractor Fine-Tuner (SFT + Evals)

## Goal
Fine-tune a model to **extract structured JSON** from unstructured text
(invoices, resumes, forms). Then **evaluate** JSON validity and field accuracy.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Structured output SFT** | Train model to produce valid JSON |
| **Evaluation harness** | Automated JSON validity + field-level accuracy |
| **Prompt-completion format** | Separate prompt/completion for loss masking |

## Stack
- `transformers` + `peft` + `trl` + `bitsandbytes`
- Model: **Qwen2.5-0.5B-Instruct**

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Create Invoice Extraction Dataset

Each example: raw invoice text → structured JSON with specific fields.
We use **prompt-completion** format so the model only learns to predict the JSON
(loss is not computed on the input text).

In [ ]:
invoice_data = [
    {"text": "Invoice #INV-2024-0891 from TechSupply Co. dated January 15, 2024. Bill to: Acme Corp, 123 Main St, Austin TX 78701. Items: 50x Laptop Stands at $29.99 each ($1,499.50), 100x USB-C Cables at $12.50 each ($1,250.00). Subtotal: $2,749.50. Tax (8.25%): $226.83. Total: $2,976.33. Due: February 14, 2024. Payment terms: Net 30.",
     "json": {"invoice_number": "INV-2024-0891", "vendor": "TechSupply Co.", "date": "2024-01-15", "bill_to": {"company": "Acme Corp", "address": "123 Main St, Austin TX 78701"}, "items": [{"description": "Laptop Stands", "quantity": 50, "unit_price": 29.99, "total": 1499.50}, {"description": "USB-C Cables", "quantity": 100, "unit_price": 12.50, "total": 1250.00}], "subtotal": 2749.50, "tax_rate": 0.0825, "tax_amount": 226.83, "total": 2976.33, "due_date": "2024-02-14", "payment_terms": "Net 30"}},

    {"text": "INVOICE Invoice No: 7742 Date: March 3, 2024 From: CloudHost Solutions, 456 Cloud Ave, Seattle WA To: DataSync Pro LLC Qty 1 - Annual Enterprise License: $24,000.00 Qty 5 - Premium Support Seats: $500 each = $2,500.00 Subtotal $26,500.00 Discount (10%): -$2,650.00 Total Due: $23,850.00 Due by: April 2, 2024 Terms: Net 30",
     "json": {"invoice_number": "7742", "vendor": "CloudHost Solutions", "date": "2024-03-03", "bill_to": {"company": "DataSync Pro LLC", "address": ""}, "items": [{"description": "Annual Enterprise License", "quantity": 1, "unit_price": 24000.00, "total": 24000.00}, {"description": "Premium Support Seats", "quantity": 5, "unit_price": 500.00, "total": 2500.00}], "subtotal": 26500.00, "tax_rate": 0.0, "tax_amount": 0.0, "total": 23850.00, "due_date": "2024-04-02", "payment_terms": "Net 30"}},

    {"text": "PrintWorks Inc. — Invoice #PW-5523 — Feb 20, 2024. Customer: Sarah's Bakery, 789 Oak Rd, Portland OR 97201. Services: 1000 Business Cards ($89.00), 500 Flyers ($175.00), Design Fee ($50.00). Subtotal: $314.00. Tax: $0.00. Grand Total: $314.00. Please pay within 15 days.",
     "json": {"invoice_number": "PW-5523", "vendor": "PrintWorks Inc.", "date": "2024-02-20", "bill_to": {"company": "Sarah's Bakery", "address": "789 Oak Rd, Portland OR 97201"}, "items": [{"description": "Business Cards", "quantity": 1000, "unit_price": 0.089, "total": 89.00}, {"description": "Flyers", "quantity": 500, "unit_price": 0.35, "total": 175.00}, {"description": "Design Fee", "quantity": 1, "unit_price": 50.00, "total": 50.00}], "subtotal": 314.00, "tax_rate": 0.0, "tax_amount": 0.0, "total": 314.00, "due_date": "", "payment_terms": "Net 15"}},

    {"text": "Invoice 10234 | Bright Electric LLC | April 10 2024 | Billed to: Johnson Property Mgmt, 55 Park Ave, Denver CO 80202 | Electrical panel upgrade: $3,200 | Wiring inspection: $450 | Parts & materials: $875 | Labor (16 hrs @ $95/hr): $1,520 | Total before tax: $6,045 | CO Sales Tax 4.5%: $272.03 | Amount Due: $6,317.03 | Due: May 10 2024",
     "json": {"invoice_number": "10234", "vendor": "Bright Electric LLC", "date": "2024-04-10", "bill_to": {"company": "Johnson Property Mgmt", "address": "55 Park Ave, Denver CO 80202"}, "items": [{"description": "Electrical panel upgrade", "quantity": 1, "unit_price": 3200.00, "total": 3200.00}, {"description": "Wiring inspection", "quantity": 1, "unit_price": 450.00, "total": 450.00}, {"description": "Parts & materials", "quantity": 1, "unit_price": 875.00, "total": 875.00}, {"description": "Labor", "quantity": 16, "unit_price": 95.00, "total": 1520.00}], "subtotal": 6045.00, "tax_rate": 0.045, "tax_amount": 272.03, "total": 6317.03, "due_date": "2024-05-10", "payment_terms": "Net 30"}},

    {"text": "FreshFood Distributors Invoice #FF-2024-331 dated 2024-05-01. Ship to: Cafe Metro, 12 Broad St, NYC 10004. 20 cases Organic Coffee Beans @ $45.00 = $900.00. 10 cases Almond Milk @ $32.00 = $320.00. 50 lbs Artisan Bread @ $4.50/lb = $225.00. Subtotal: $1,445.00. Delivery: $75.00. Tax (8.875%): $134.89. Invoice Total: $1,654.89. Terms: COD.",
     "json": {"invoice_number": "FF-2024-331", "vendor": "FreshFood Distributors", "date": "2024-05-01", "bill_to": {"company": "Cafe Metro", "address": "12 Broad St, NYC 10004"}, "items": [{"description": "Organic Coffee Beans", "quantity": 20, "unit_price": 45.00, "total": 900.00}, {"description": "Almond Milk", "quantity": 10, "unit_price": 32.00, "total": 320.00}, {"description": "Artisan Bread", "quantity": 50, "unit_price": 4.50, "total": 225.00}], "subtotal": 1445.00, "tax_rate": 0.08875, "tax_amount": 134.89, "total": 1654.89, "due_date": "", "payment_terms": "COD"}},

    {"text": "MEGA SOFTWARE SOLUTIONS — INV-MS-9001 — 06/15/2024. To: GlobalTech Industries, 1 Innovation Dr, San Jose CA 95134. Custom CRM Development: $45,000. API Integration Module: $12,000. 12-month maintenance contract: $8,400. Cloud hosting setup: $3,600. Total: $69,000. No tax (B2B exempt). Payment: 50% upfront, 50% on delivery.",
     "json": {"invoice_number": "INV-MS-9001", "vendor": "Mega Software Solutions", "date": "2024-06-15", "bill_to": {"company": "GlobalTech Industries", "address": "1 Innovation Dr, San Jose CA 95134"}, "items": [{"description": "Custom CRM Development", "quantity": 1, "unit_price": 45000.00, "total": 45000.00}, {"description": "API Integration Module", "quantity": 1, "unit_price": 12000.00, "total": 12000.00}, {"description": "12-month maintenance contract", "quantity": 1, "unit_price": 8400.00, "total": 8400.00}, {"description": "Cloud hosting setup", "quantity": 1, "unit_price": 3600.00, "total": 3600.00}], "subtotal": 69000.00, "tax_rate": 0.0, "tax_amount": 0.0, "total": 69000.00, "due_date": "", "payment_terms": "50% upfront, 50% on delivery"}},

    {"text": "Quick Clean Services Invoice 4455 July 8, 2024 Client: River View Apartments 200 Harbor Blvd Chicago IL 60601 Monthly cleaning - 20 units @ $85: $1,700.00 Deep clean - 3 units @ $250: $750.00 Carpet shampooing - 5 units @ $150: $750.00 Supplies: $125.00 Subtotal: $3,325.00 IL Tax 6.25%: $207.81 Total: $3,532.81 Net 15",
     "json": {"invoice_number": "4455", "vendor": "Quick Clean Services", "date": "2024-07-08", "bill_to": {"company": "River View Apartments", "address": "200 Harbor Blvd Chicago IL 60601"}, "items": [{"description": "Monthly cleaning", "quantity": 20, "unit_price": 85.00, "total": 1700.00}, {"description": "Deep clean", "quantity": 3, "unit_price": 250.00, "total": 750.00}, {"description": "Carpet shampooing", "quantity": 5, "unit_price": 150.00, "total": 750.00}, {"description": "Supplies", "quantity": 1, "unit_price": 125.00, "total": 125.00}], "subtotal": 3325.00, "tax_rate": 0.0625, "tax_amount": 207.81, "total": 3532.81, "due_date": "", "payment_terms": "Net 15"}},

    {"text": "Digital Marketing Hub - Invoice DM-8899 - Aug 1 2024 - For: PetPal App Inc, 88 Startup Ln, Miami FL 33101 - Google Ads management (Aug): $2,000 - Social media content (20 posts): $1,500 - SEO audit & optimization: $3,000 - Email campaign design: $800 - Before tax: $7,300 - FL Sales Tax 7%: $511.00 - Total: $7,811.00 - Due: Aug 31 2024",
     "json": {"invoice_number": "DM-8899", "vendor": "Digital Marketing Hub", "date": "2024-08-01", "bill_to": {"company": "PetPal App Inc", "address": "88 Startup Ln, Miami FL 33101"}, "items": [{"description": "Google Ads management", "quantity": 1, "unit_price": 2000.00, "total": 2000.00}, {"description": "Social media content", "quantity": 20, "unit_price": 75.00, "total": 1500.00}, {"description": "SEO audit & optimization", "quantity": 1, "unit_price": 3000.00, "total": 3000.00}, {"description": "Email campaign design", "quantity": 1, "unit_price": 800.00, "total": 800.00}], "subtotal": 7300.00, "tax_rate": 0.07, "tax_amount": 511.00, "total": 7811.00, "due_date": "2024-08-31", "payment_terms": "Net 30"}},
]

# Hold out last 2 for evaluation
train_data = invoice_data[:6]
eval_data = invoice_data[6:]
print(f"Train: {len(train_data)} examples, Eval: {len(eval_data)} examples")

In [ ]:
from datasets import Dataset

SYSTEM_MSG = "You are a structured data extractor. Given raw invoice text, output a valid JSON object with these fields: invoice_number, vendor, date (YYYY-MM-DD), bill_to (company, address), items (list of description, quantity, unit_price, total), subtotal, tax_rate, tax_amount, total, due_date, payment_terms. Output ONLY valid JSON, no explanation."

def format_example(ex):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": f"Extract structured data from this invoice:\n\n{ex['text']}"},
            {"role": "assistant", "content": json.dumps(ex["json"], indent=2)},
        ]
    }

train_dataset = Dataset.from_list([format_example(ex) for ex in train_data])
print(f"Training dataset: {train_dataset}")

## Step 2 — Fine-Tune

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./json_extractor_model"

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_steps=80,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=8,
    max_length=1024,       # invoices can be long
    bf16=True,
    logging_steps=10,
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=MODEL_ID,
    args=sft_config,
    train_dataset=train_dataset,
    peft_config=lora_config,
)

print("Starting JSON extractor SFT training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("\n✅ Training complete!")

## Step 3 — Evaluation Harness

We evaluate on 3 axes:
1. **JSON validity** — does the output parse as JSON?
2. **Schema completeness** — are all expected fields present?
3. **Field accuracy** — do extracted values match ground truth?

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

REQUIRED_FIELDS = ["invoice_number", "vendor", "date", "bill_to", "items", "subtotal", "total"]

def extract_json(text):
    """Run the model and attempt to parse JSON from output."""
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": f"Extract structured data from this invoice:\n\n{text}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=600, temperature=0.1, do_sample=True)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return raw

def evaluate_extraction(raw_output, expected):
    """Score an extraction on validity, completeness, and accuracy."""
    result = {"json_valid": False, "schema_complete": False, "field_accuracy": 0.0, "errors": []}

    # 1. JSON validity
    try:
        parsed = json.loads(raw_output)
        result["json_valid"] = True
    except json.JSONDecodeError as e:
        result["errors"].append(f"Invalid JSON: {e}")
        return result

    # 2. Schema completeness
    missing = [f for f in REQUIRED_FIELDS if f not in parsed]
    result["schema_complete"] = len(missing) == 0
    if missing:
        result["errors"].append(f"Missing fields: {missing}")

    # 3. Field accuracy (check key scalar fields)
    checks = 0
    matches = 0
    for field in ["invoice_number", "vendor", "total"]:
        if field in parsed and field in expected:
            checks += 1
            if str(parsed[field]).strip().lower() == str(expected[field]).strip().lower():
                matches += 1
            else:
                result["errors"].append(f"{field}: got '{parsed[field]}', expected '{expected[field]}'")

    result["field_accuracy"] = matches / checks if checks > 0 else 0.0
    return result

print("Evaluation harness ready")

In [ ]:
# Run evaluation on held-out examples
print("Running evaluation on held-out invoices...\n")

all_results = []
for i, ex in enumerate(eval_data):
    raw = extract_json(ex["text"])
    score = evaluate_extraction(raw, ex["json"])
    all_results.append(score)

    print(f"Invoice {i+1}:")
    print(f"  JSON Valid:     {'✅' if score['json_valid'] else '❌'}")
    print(f"  Schema OK:      {'✅' if score['schema_complete'] else '❌'}")
    print(f"  Field Accuracy: {score['field_accuracy']:.0%}")
    if score["errors"]:
        for err in score["errors"]:
            print(f"  ⚠️  {err}")
    print()

# Summary
valid_pct = sum(1 for r in all_results if r["json_valid"]) / len(all_results) * 100
complete_pct = sum(1 for r in all_results if r["schema_complete"]) / len(all_results) * 100
avg_acc = sum(r["field_accuracy"] for r in all_results) / len(all_results) * 100

print("=" * 50)
print(f"JSON Validity:      {valid_pct:.0f}%")
print(f"Schema Completeness: {complete_pct:.0f}%")
print(f"Avg Field Accuracy:  {avg_acc:.0f}%")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Structured output SFT** | Train on (text → JSON) pairs |
| **System prompt** | Defines exact JSON schema to follow |
| **3-axis eval** | JSON validity, schema completeness, field accuracy |
| **Train/eval split** | 6 train, 2 eval — always hold out data |

## 🔧 Production Extensions
- Use 500+ real invoices with OCR-extracted text
- Add constrained decoding (e.g., `outlines` library) to guarantee valid JSON
- Evaluate items array with fuzzy matching
- Add confidence scores per field
- Compare vs. regex-based extraction for cost/accuracy tradeoff